# 01. Lokalny LLM w Google Colab: Qwen 0.5B Instruct

Ten notebook pokazuje, jak uruchomić mały model językowy bez użycia zewnętrznego API.

**Cel zajęć:** student rozumie, że LLM może działać jako model w środowisku wykonawczym, a nie tylko jako usługa chmurowa.

**Zalecenie:** w Colabie wybierz `Runtime → Change runtime type → T4 GPU`. Notebook zadziała też na CPU, ale wolniej.

In [1]:
!pip -q install transformers accelerate sentencepiece


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
print("PyTorch:", torch.__version__)
print("GPU dostępne:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
GPU dostępne: True
GPU: NVIDIA GeForce RTX 5070


## 1. Ładowanie modelu

Używamy małego modelu `Qwen/Qwen2.5-0.5B-Instruct`, ponieważ nadaje się do szybkiego pokazu dydaktycznego w Colabie.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Urządzenie:", device)

# Uwaga: na GPU używamy float16, na CPU float32.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
model.to(device)
model.eval()

print("Model załadowany:", MODEL_NAME)

/home/jakub/PycharmProjects/SWPS_2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Urządzenie: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3507.19it/s]


Model załadowany: Qwen/Qwen2.5-0.5B-Instruct


## 2. Funkcja `ask_llm()`

Ta funkcja tworzy prosty interfejs: prompt → odpowiedź modelu.

In [4]:
def ask_llm(prompt, system="Jesteś pomocnym asystentem dydaktycznym. Odpowiadasz po polsku, krótko i precyzyjnie.", max_new_tokens=250):
    """Prosta funkcja do rozmowy z lokalnym modelem uruchomionym w Colabie."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip()

In [5]:
answer = ask_llm("Wyjaśnij studentom, czym różni się lokalny LLM od modelu w chmurze.")
print(answer)

Localized Machine Learning (LLM) to system, który jest używany do przetwarzania danych lokalnych lub lokalnych informacji. To jest szczególnie ważne dla wielu zadań, które wymagają dostępu do lokalnych danych, takich jak bazy danych lokalne, analiza lokalnej historii danych, czy też analiza lokalnych danych na platformie lokalnym.

Model w chmurze, na konkretnych przypadkach, może być używany do przetwarzania danych globalnych lub globalnych danych, co może być niebezpieczne lub niepotrzebne dla lokalnych danych. 

Warto pamiętać, że lokalna lokalność może mieć znaczenie dla różnych zastosowań, takich jak:

1. Analiza lokalnych danych: W tym przypadku warto używać lokalnego modelu, aby uzyskać najlepsze wyniki.
2. Analiza lokalnych danych na platformie lokalnym: W tym przypadku warto używać lokalnego modelu, aby uzyskać najlepsze wyniki.
3. Analiza lokalnych danych na platformie lokalnym: W tym przypadku warto używać lokal


## 3. Ten sam temat, różne style odbiorcy

In [6]:
prompts = [
    "Wyjaśnij lokalne LLM-y studentowi pierwszego roku.",
    "Wyjaśnij lokalne LLM-y administratorowi IT.",
    "Wyjaśnij lokalne LLM-y dyrektorowi firmy."
]

for p in prompts:
    print("=" * 90)
    print("PROMPT:", p)
    print(ask_llm(p, max_new_tokens=220))

PROMPT: Wyjaśnij lokalne LLM-y studentowi pierwszego roku.
Pamiętaj, że lokalne LLM (Master of Laws) studentowi to nie tylko termin, ale także konsekwencje, które mogą być związane z jego studią lub przedstawieniem się jako profesor. Warto pamiętać, że:

1. **Przedsiębiorstwo**: Jeśli student jest pracownikiem lub profesorem, może to znaczyć, że ma możliwość pracy w różnych firmach, krajowych firm, czy evenziu.

2. **Zastosowanie na wykładowce**: Możesz być przekazany do wykładowców lub profesjonalistów, aby pomóc w sprawie nauki, prace lub projektów.

3. **Przepraszanie dla innych**: Jeśli student jest przeszkodowany lub nie jest w stanie pracować, może to znaczyć, że ma możliwość przepraszania dla innych.

4. **Przewidywane zainteresowania**: Może to być konkretna zaint
PROMPT: Wyjaśnij lokalne LLM-y administratorowi IT.
Localny LMDB (Local Memory Database) – to system zasobów pamięci lokalnych, które są używane przez aplikacje w środowisku lokalnym. Wartości są dostosowane do lokaln

## 4. Klasyfikacja intencji przez LLM

To pierwszy krok do workflow agentowego: model nie tylko odpowiada, ale pomaga wybrać ścieżkę działania.

In [ ]:
def classify_intent_with_llm(user_question):
    prompt = f"""
Zaklasyfikuj intencję użytkownika do jednej z kategorii:

PROMPT_ONLY - wystarczy zwykłe pytanie do modelu
RAG - potrzebne są dokumenty lub baza wiedzy
TRAINING - potrzebny jest trening albo dostrajanie modelu
TOOL - potrzebne jest narzędzie zewnętrzne, np. Python, kalkulator, baza danych

Pytanie użytkownika:
{user_question}

Odpowiedz dokładnie w takim formacie:
KATEGORIA: ...
UZASADNIENIE: ...
"""
    return ask_llm(prompt, max_new_tokens=150)

test_questions = [
    "Wyjaśnij mi, czym jest transformer.",
    "Mam dokument PDF i chcę zadawać pytania o jego treść.",
    "Policz średnią sprzedaż z pliku CSV.",
    "Chcę nauczyć model mojego stylu pisania."
]

for q in test_questions:
    print("=" * 90)
    print("PYTANIE:", q)
    print(classify_intent_with_llm(q))

## 5. Wariant zaawansowany: LM Studio

Colab nie widzi `localhost` Twojego komputera. Dlatego `http://localhost:1234/v1` zadziała lokalnie na laptopie, ale nie z Colaba.

Poniższa komórka jest tylko szablonem, jeśli ktoś wystawi LM Studio przez publiczny tunel.

In [ ]:
# Opcjonalny szablon. Nie uruchamiaj bez publicznego adresu LM Studio.
# !pip -q install openai
# from openai import OpenAI
#
# client = OpenAI(
#     base_url="TU_PUBLICZNY_ADRES_LM_STUDIO/v1",
#     api_key="lm-studio"
# )
#
# response = client.chat.completions.create(
#     model="local-model",
#     messages=[{"role": "user", "content": "Wyjaśnij, czym jest lokalny LLM."}],
#     temperature=0.2
# )
#
# print(response.choices[0].message.content)

## Puenta dydaktyczna

Model jest silnikiem. Workflow jest pojazdem. Sam silnik nie wie, czy ma odpowiadać z dokumentów, liczyć w Pythonie, czy tylko wyjaśnić pojęcie.